<a href="https://colab.research.google.com/github/Pauaza/PBL_Sistem-Cerdas-Rencana-Riset/blob/uts-sbp/data_cleaning_%26_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import re

def clean_text(text):
    """Fungsi pembersihan teks"""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) # Menjaga huruf dan angka
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 1. Load Dataset
df_lab = pd.read_csv('lab_dosen.csv')
df_skripsi = pd.read_csv('skripsi_alumni.csv')
df_penelitian = pd.read_csv('penelitian_dosen.csv')

# --- NORMALISASI NAMA KOLOM & CLEANING AWAL ---
# Lab: Ubah 'dosen_name' menjadi 'nama_dosen'
df_lab = df_lab.rename(columns={'dosen_name': 'nama_dosen'})
df_lab['nama_dosen'] = df_lab['nama_dosen'].str.strip()

# Penelitian: Pastikan teks bersih sebelum digabung
df_penelitian['nama_dosen'] = df_penelitian['nama_dosen'].str.strip()
df_penelitian['judul_penelitian'] = df_penelitian['judul_penelitian'].apply(clean_text)
df_penelitian['abstrak_penelitian'] = df_penelitian['abstrak_penelitian'].apply(clean_text)

# Skripsi: Normalisasi nama p1 dan p2
df_skripsi = df_skripsi.rename(columns={'judul': 'judul_skripsi'})
df_skripsi['judul_skripsi'] = df_skripsi['judul_skripsi'].apply(clean_text)

In [18]:
# --- TAHAP 1: AGREGASI PENELITIAN (1 Dosen = 1 Baris) ---
# Menggabungkan semua riwayat penelitian menjadi satu baris panjang per dosen
df_penelitian_grouped = df_penelitian.groupby('nama_dosen').agg({
    'judul_penelitian': lambda x: ' | '.join(x),
    'abstrak_penelitian': lambda x: ' | '.join(x),
    'tahun_publikasi': lambda x: ', '.join(map(str, x))
}).reset_index()

In [19]:
# --- TAHAP 2: TRANSFORMASI & AGREGASI SKRIPSI ---
# Ubah format 'menyamping' (p1, p2) menjadi 'memanjang' (nama_dosen)
df_skripsi_long = pd.melt(
    df_skripsi,
    id_vars=['judul_skripsi'],
    value_vars=['p1', 'p2'],
    value_name='nama_dosen'
).dropna(subset=['nama_dosen'])

df_skripsi_long['nama_dosen'] = df_skripsi_long['nama_dosen'].str.strip()

# Gabungkan semua judul skripsi alumni yang pernah dibimbing dosen tersebut
df_skripsi_grouped = df_skripsi_long.groupby('nama_dosen').agg({
    'judul_skripsi': lambda x: ' | '.join(x)
}).reset_index()

In [20]:
# --- TAHAP 3: MERGING FINAL (Penyatuan Profil) ---
# Menggabungkan data Lab, Penelitian, dan Skripsi berdasarkan Nama Dosen
df_final = pd.merge(df_lab, df_penelitian_grouped, on='nama_dosen', how='outer')
df_final = pd.merge(df_final, df_skripsi_grouped, on='nama_dosen', how='outer')

# Menghapus baris jika nama_dosen kosong atau tidak valid
df_final.dropna(subset=['nama_dosen'], inplace=True)

In [21]:
# --- TAHAP 4: ID INCREMENT BERDASARKAN ALFABET ---
df_final.sort_values(by='nama_dosen', ascending=True, inplace=True)
df_final.reset_index(drop=True, inplace=True)
df_final.insert(0, 'id_dosen_master', df_final.index + 1)

# Simpan ke Master Dataset
df_final.to_csv('master_dataset_dosen.csv', index=False)

print("✅ Master Dataset Berhasil Dibuat!")
print(f"Total Dosen Unik yang Terdaftar: {len(df_final)}")

✅ Master Dataset Berhasil Dibuat!
Total Dosen Unik yang Terdaftar: 56
